In [25]:
import sys
sys.path.append('../src')

import sqlite3
import pandas as pd
from config import DB_PATH, RAW_TABLES

conn = sqlite3.connect(DB_PATH)
pd.set_option('display.max_columns', None)

In [28]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables  

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers
8,category_translation
9,data_dictionary


In [29]:
dq_report = pd.read_sql("SELECT * FROM dq_report", conn)

dq_report[dq_report['null_pct'] > 0]

,table_name,column_name,null_pct,duplicate_rows_in_table,row_count
25,order_reviews,review_comment_title,88.34,0,99224
26,order_reviews,review_comment_message,58.70,0,99224
33,orders,order_approved_at,0.16,0,99441
34,orders,order_delivered_carrier_date,1.79,0,99441
35,orders,order_delivered_customer_date,2.98,0,99441
38,products,product_category_name,1.85,0,32951
39,products,product_name_lenght,1.85,0,32951
40,products,product_description_lenght,1.85,0,32951
41,products,product_photos_qty,1.85,0,32951
42,products,product_weight_g,0.01,0,32951


In [30]:
for table_name in RAW_TABLES:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    print(f"\n{'='*60}\n{table_name.upper()}  —  shape: {df.shape}\n{'='*60}")
    print(df.dtypes)
    print("\nNull %:")
    print((df.isna().sum() / len(df) * 100).round(2))


CUSTOMERS  —  shape: (99441, 5)
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Null %:
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

GEOLOCATION  —  shape: (1000163, 5)
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Null %:
geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64

ORDER_ITEMS  —  shape: (112650, 7)
order_id                object
order_item_id            int64
product_id              object
seller_id           

In [31]:
from data_profiling import ProfileReport

# Pick one table to start with — orders is a good central one
orders_df = pd.read_sql("SELECT * FROM orders", conn)

profile = ProfileReport(orders_df, title="Orders Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 8/8 [00:04<00:00,  1.71it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

**What % of orders never got delivered, and does that correlate with specific sellers, categories, or regions?**

In [32]:
orders_df['missing_carrier_date'] = orders_df['order_delivered_carrier_date'].isna()
orders_df['missing_customer_date'] = orders_df['order_delivered_customer_date'].isna()

status_breakdown = orders_df.groupby('order_status').agg(
    total_orders=('order_id', 'count'),
    missing_carrier_date=('missing_carrier_date', 'sum'),
    missing_customer_date=('missing_customer_date', 'sum')
).reset_index()

status_breakdown

,order_status,total_orders,missing_carrier_date,missing_customer_date
0,approved,2,2,2
1,canceled,625,550,619
2,created,5,5,5
3,delivered,96478,2,8
4,invoiced,314,314,314
5,processing,301,301,301
6,shipped,1107,0,1107
7,unavailable,609,609,609


In [33]:
delivered_but_missing = orders_df[
    (orders_df['order_status'] == 'delivered') &
    (orders_df['missing_carrier_date'] | orders_df['missing_customer_date'])
]

delivered_but_missing[[
    'order_id', 'order_status', 'order_purchase_timestamp',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23,None,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00,None,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00,None,2018-07-30 00:00:00
73222,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,None,2017-11-20 19:44:47,2017-11-14 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00,None,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00,None,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,None,None,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-12 14:10:00,None,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-03 19:26:00,None,2018-07-19 00:00:00


**These 9 orders are logged as "delivered" in order_status, but have inconsistent or missing delivery timestamps — most commonly the carrier date is recorded but the customer delivery date is not. This affects <0.01% of delivered orders and is treated as a data logging inconsistency, not excluded from delivery-time analysis (since excluding them wouldn't materially change results) but explicitly flagged in the data quality section.**

In [34]:
order_items_df = pd.read_sql("SELECT * FROM order_items", conn)

profile = ProfileReport(order_items_df, title="Order Items Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:01<00:00,  3.79it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
order_items_df.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [36]:
zero_freight = order_items_df[order_items_df['freight_value'] == 0]
print(f"Number of rows with $0 freight: {len(zero_freight)}")
zero_freight.head()

Number of rows with $0 freight: 383


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0


In [37]:
zero_freight['seller_id'].value_counts().head(10)

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64

**$0 freight_value is not randomly distributed — 96% of occurrences trace back to 4 specific sellers (out of 3,095 total), suggesting a seller-specific shipping arrangement (e.g., local pickup, bundled shipping, or free-shipping promotions) rather than a data quality defect. Affects 0.34% of order_items and does not require imputation**

In [38]:
products_df = pd.read_sql("SELECT * FROM products", conn)

profile = ProfileReport(products_df, title="Products Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 9/9 [00:00<00:00, 22.87it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [39]:
missing_all = products_df[
    products_df['product_category_name'].isna() &
    products_df['product_name_lenght'].isna() &
    products_df['product_description_lenght'].isna() &
    products_df['product_photos_qty'].isna()
]
print(f"Rows missing all 4 fields together: {len(missing_all)}")
missing_all.head()

Rows missing all 4 fields together: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,None,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,None,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,None,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,None,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,None,NaN,NaN,NaN,300.0,35.0,7.0,12.0


**610 products (1.9%) are missing category, name length, description length, and photo count together but retain their physical shipping attributes (weight, dimensions). This suggests incomplete seller-submitted listings (marketing/descriptive fields skipped) rather than corrupted records, since the products are still shippable and were evidently sold. Not imputed; flagged as incomplete metadata.**

In [40]:
reviews_df = pd.read_sql("SELECT * FROM order_reviews", conn)

profile = ProfileReport(reviews_df, title="Order Reviews Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:03<00:00,  2.00it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [41]:
print(reviews_df['review_score'].isna().sum())
reviews_df['review_score'].value_counts().sort_index()

0


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

**review_score has zero missing values across all 99,224 reviews and shows a strong positive skew (77% are 4-5 stars). Review_comment_title (88.3%) and review_comment_message (58.7%) are optional free-text fields with high missingness — expected behavior, not a data quality defect, and their absence doesn't affect the core rating field used for quantitative analysis**

In [42]:
geo_df = pd.read_sql("SELECT * FROM geolocation", conn)

profile = ProfileReport(geo_df, title="Geolocation Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:01<00:00,  2.91it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [43]:
print("Total rows:", len(geo_df))
print("Unique zip codes:", geo_df['geolocation_zip_code_prefix'].nunique())
print("\nLat/Lng range check:")
print(geo_df[['geolocation_lat', 'geolocation_lng']].describe())

Total rows: 1000163
Unique zip codes: 19015

Lat/Lng range check:
       geolocation_lat  geolocation_lng
count     1.000163e+06     1.000163e+06
mean     -2.117615e+01    -4.639054e+01
std       5.715866e+00     4.269748e+00
min      -3.660537e+01    -1.014668e+02
25%      -2.360355e+01    -4.857317e+01
50%      -2.291938e+01    -4.663788e+01
75%      -1.997962e+01    -4.376771e+01
max       4.506593e+01     1.211054e+02


Lat/Lng range — genuine outliers found
Brazil's actual coordinates roughly span latitude -34 to 5, and longitude -74 to -34. But your data shows:

max latitude: 45.07 — this is nowhere near Brazil (that's roughly Europe/Central Asia latitude)
max longitude: 121.11 — this is in Asia (nowhere close to South America)
min longitude: -101.47 — this is around Mexico/Central America, still outside Brazil

In [44]:
#Isolate these bad coordinates
bad_coords = geo_df[
    (geo_df['geolocation_lat'] > 6) | (geo_df['geolocation_lat'] < -34) |
    (geo_df['geolocation_lng'] > -34) | (geo_df['geolocation_lng'] < -74)
]
print(f"Rows with coordinates outside Brazil's bounds: {len(bad_coords)}")
bad_coords.head()

Rows with coordinates outside Brazil's bounds: 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ


In [45]:
bad_coords[['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
387565,18243,bom retiro da esperanca,SP
513631,28165,vila nova de campos,RJ
513643,28155,santa maria,RJ
513754,28155,santa maria,RJ
514429,28333,raposo,RJ
516682,28595,portela,RJ
538512,29654,santo antônio do canaã,ES
538557,29654,santo antonio do canaa,ES
585242,35179,santana do paraíso,MG
585260,35179,santana do paraiso,MG


**&1,000,163 rows map to only 19,015 unique zip codes (~53 coordinate samples per zip); will be aggregated before joining to customers/sellers. 42 rows (0.004%) have lat/lng coordinates falling outside Brazil's bounds despite valid city/state/zip — likely geocoding errors, to be replaced with the zip code's average coordinate. Minor accent/spelling inconsistencies also found in city names for the same zip code.**

In [46]:
customers_df = pd.read_sql("SELECT * FROM customers", conn)

profile = ProfileReport(customers_df, title="Customers Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:02<00:00,  2.11it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [47]:
print("Total rows:", len(customers_df))
print("Unique customer_id:", customers_df['customer_id'].nunique())
print("Unique customer_unique_id:", customers_df['customer_unique_id'].nunique())


Total rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096


**customers table has two ID columns: customer_id (unique per order — 99,441 total) and customer_unique_id (unique per actual person — 96,096 total). The difference (3,345) represents repeat customers. All customer-level aggregations (e.g., repeat purchase rate, customer lifetime value) must use customer_unique_id, not customer_id, to avoid double-counting returning customers.**

In [48]:
sellers_df = pd.read_sql("SELECT * FROM sellers", conn)

profile = ProfileReport(sellers_df, title="Sellers Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 4/4 [00:00<00:00, 106.73it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [49]:
print("Total sellers:", len(sellers_df))
print(sellers_df['seller_state'].value_counts())


Total sellers: 3095
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
PI       1
AC       1
MA       1
AM       1
PA       1
Name: count, dtype: int64


**Sellers are heavily concentrated geographically — SP alone accounts for 59.7% of all 3,095 sellers, with the top 3 states (SP, PR, MG) covering 79%. Several states have only 1 seller. This imbalance must be accounted for in any state-level seller performance comparison (e.g., using rates/medians rather than raw counts, or excluding states with negligible seller counts from statistical comparisons).**

In [50]:
payments_df = pd.read_sql("SELECT * FROM order_payments", conn)

profile = ProfileReport(payments_df, title="Order Payments Table Profiling Report", minimal=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:00<00:00,  7.34it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [51]:
print("Total rows:", len(payments_df))
print("Unique order_id:", payments_df['order_id'].nunique())
print("\nPayment type breakdown:")
print(payments_df['payment_type'].value_counts())
print("\nInstallments range:", payments_df['payment_installments'].min(), "to", payments_df['payment_installments'].max())

Total rows: 103886
Unique order_id: 99440

Payment type breakdown:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Installments range: 0 to 24


In [52]:
zero_installments = payments_df[payments_df['payment_installments'] == 0]
print(f"Rows with 0 installments: {len(zero_installments)}")
zero_installments['payment_type'].value_counts()

Rows with 0 installments: 2


payment_type
credit_card    2
Name: count, dtype: int64

**103,886 payment records map to 99,440 unique orders — 4,446 orders used multiple payment methods (e.g., split between voucher and credit card). Order-level payment totals must be aggregated (SUM by order_id), not counted as rows. payment_type includes 3 rows with an undefined/unknown category ('not_defined') and payment_installments has 2 rows with a value of 0 despite being credit_card payments — both negligible in scale (<0.01% combined) and excluded from installment-based analysis without further correction**